# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MissNaliaka/SEO-Content-Opportunity-Scoring/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [1]:
import duckdb
import numpy as np
import pandas as pd
from google.colab import userdata

con = duckdb.connect()
hf_token = userdata.get("PurpleElegantBass749671")
print(f"Token loaded: {hf_token[:6]}... (length {len(hf_token)})" if hf_token else "Token is empty/None!")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

rel = "hf://datasets/FlyRank/internship-warehouse"
DECISION_MOMENT = "2026-04-30"
WINDOW_START = "2026-01-30"
SPLIT_DATE = "2026-03-15"

#read the daily content performance data from Parquet files covering January through April 2026
#filter it to the specific feature window between WINDOW_START and DECISION_MOMENT
#group the data by client and page
#split impressions into an early period (imp_early) and a later period (imp_late) based on SPLIT_DATE
#calculates total impressions, total clicks, and average search position for the entire window
#produce a DataFrame with one row per client-page combination

feature_paths = [f"{rel}/fact_content_daily_performance/month=2026-0{m}/*.parquet" for m in [1, 2, 3, 4]]

feature_df = con.sql(f"""
    WITH daily AS (
        SELECT client_hash_id, content_hash_id, report_date,
               gsc_impressions, gsc_clicks, gsc_avg_position
        FROM read_parquet([{', '.join(f"'{p}'" for p in feature_paths)}])
        WHERE report_date BETWEEN DATE '{WINDOW_START}' AND DATE '{DECISION_MOMENT}'
    )


    SELECT
        client_hash_id, content_hash_id,
        SUM(CASE WHEN report_date <= DATE '{SPLIT_DATE}' THEN gsc_impressions ELSE 0 END) AS imp_early,
        SUM(CASE WHEN report_date >  DATE '{SPLIT_DATE}' THEN gsc_impressions ELSE 0 END) AS imp_late,
        SUM(gsc_impressions) AS impressions_90d,
        SUM(gsc_clicks) AS clicks_90d,
        AVG(gsc_avg_position) AS avg_position_90d
    FROM daily
    GROUP BY client_hash_id, content_hash_id
""").df()

qualifying = con.sql(f"""
    SELECT c.content_hash_id, c.client_hash_id, c.content_type, c.main_intent,
           c.word_count, c.char_count, c.competition_level, c.search_volume
    FROM read_parquet('{rel}/dim_content.parquet') c
    JOIN read_parquet('{rel}/dim_clients.parquet') cl USING (client_hash_id)
    WHERE c.content_created_date <= DATE '{WINDOW_START}'
      AND cl.gsc_data_start <= DATE '{WINDOW_START}'
""").df()

feature_df = feature_df.merge(qualifying, on=["client_hash_id", "content_hash_id"], how="inner")
feature_df["ctr"] = np.where(
    feature_df["impressions_90d"] > 0,
    (feature_df["clicks_90d"] / feature_df["impressions_90d"]) * 100,
    np.nan,
)
feature_df["is_declining"] = (feature_df["imp_late"] < feature_df["imp_early"]).astype(int)

print(f"Rows: {len(feature_df):,}")
feature_df.head()

Token loaded: hf_IcB... (length 37)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 260,285


,client_hash_id,content_hash_id,imp_early,imp_late,impressions_90d,clicks_90d,avg_position_90d,content_type,main_intent,word_count,char_count,competition_level,search_volume,ctr,is_declining
0,client_3ffa76342f366962,content_fb7fc66912a05d77,2.0,0.0,2.0,0.0,3.0,feedly article,None,1012,7423,None,<NA>,0.0,1
1,client_3ffa76342f366962,content_9715d495c88cac04,1.0,0.0,1.0,0.0,7.0,feedly article,None,781,5620,None,<NA>,0.0,1
2,client_3ffa76342f366962,content_7fa9ba77cc2e4f6a,0.0,0.0,0.0,0.0,NaN,feedly article,None,729,5400,None,<NA>,NaN,0
3,client_3ffa76342f366962,content_17256d7aafd67692,0.0,0.0,0.0,0.0,NaN,feedly article,None,835,6182,None,<NA>,NaN,0
4,client_3ffa76342f366962,content_5adb3fbc947e4fbe,0.0,0.0,0.0,0.0,NaN,feedly article,None,893,6302,None,<NA>,NaN,0


In [2]:
may_df = con.sql(f"""
    SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS impressions_may
    FROM read_parquet('{rel}/fact_content_daily_performance/month=2026-05/*.parquet')
    GROUP BY client_hash_id, content_hash_id
""").df()

model_df = feature_df.merge(may_df, on=["client_hash_id", "content_hash_id"], how="left")
model_df["impressions_may"] = model_df["impressions_may"].fillna(0)

LATE_DAYS, MAY_DAYS = 46, 31
model_df["rate_late"] = model_df["imp_late"] / LATE_DAYS
model_df["rate_may"] = model_df["impressions_may"] / MAY_DAYS
model_df["still_declining_may"] = (model_df["rate_may"] < 0.8 * model_df["rate_late"]).astype(int)

print(f"Rows: {len(model_df):,}")
print(f"still_declining_may base rate: {model_df['still_declining_may'].mean():.3f}")

numeric_features = ["imp_early", "imp_late", "impressions_90d", "clicks_90d", "avg_position_90d", "ctr",
                     "word_count", "char_count", "search_volume"]
categorical_features = ["content_type", "main_intent", "competition_level"]
banned = {"is_declining", "impressions_may", "rate_late", "rate_may", "still_declining_may",
          "content_hash_id", "client_hash_id"}
assert not (set(numeric_features) | set(categorical_features)) & banned, "leakage/grouping column in feature list"
print(f"Numeric features: {len(numeric_features)}  |  Categorical features: {len(categorical_features)}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 260,285
still_declining_may base rate: 0.289
Numeric features: 9  |  Categorical features: 3


In [3]:
RANDOM_STATE = 42
N_FOLDS = 5

client_sizes = model_df.groupby("client_hash_id").size().sort_values(ascending=False)
fold_totals = np.zeros(N_FOLDS, dtype=int)
client_to_fold = {}
for client, size in client_sizes.items():
    smallest_fold = int(np.argmin(fold_totals))
    client_to_fold[client] = smallest_fold
    fold_totals[smallest_fold] += size

print("Rows per fold after size-balancing:")
for f in range(N_FOLDS):
    n_clients = sum(1 for v in client_to_fold.values() if v == f)
    print(f"  fold {f}: {fold_totals[f]:>7,} rows across {n_clients} clients")

model_df["fold"] = model_df["client_hash_id"].map(client_to_fold)

Rows per fold after size-balancing:
  fold 0:  52,005 rows across 7 clients
  fold 1:  52,065 rows across 9 clients
  fold 2:  52,165 rows across 7 clients
  fold 3:  51,998 rows across 9 clients
  fold 4:  52,052 rows across 9 clients


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

def build_matrix(frame, columns=None):
    num = frame[numeric_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    cat = frame[categorical_features].fillna("unknown").astype(str)
    enc = pd.get_dummies(cat, prefix=categorical_features, dtype=float)
    mat = pd.concat([num.reset_index(drop=True), enc.reset_index(drop=True)], axis=1)
    if columns is not None:
        mat = mat.reindex(columns=columns, fill_value=0)
    return mat

y = model_df["still_declining_may"].to_numpy()

model_builders = {
    "logistic_regression": lambda: Pipeline([
        ("scaler", StandardScaler()),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "decision_tree": lambda: DecisionTreeClassifier(
        max_depth=4, min_samples_leaf=200, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "random_forest": lambda: RandomForestClassifier(
        n_estimators=300, max_depth=8, min_samples_leaf=50,
        class_weight="balanced_subsample", random_state=RANDOM_STATE, n_jobs=-1
    ),
}

oof_scores = {name: np.zeros(len(model_df)) for name in model_builders}
for f in range(N_FOLDS):
    train_mask = model_df["fold"] != f
    test_mask = model_df["fold"] == f
    X_train = build_matrix(model_df[train_mask])
    X_test = build_matrix(model_df[test_mask], columns=X_train.columns)
    y_train = y[train_mask.to_numpy()]
    for name, builder in model_builders.items():
        model = builder()
        model.fit(X_train, y_train)
        oof_scores[name][test_mask.to_numpy()] = model.predict_proba(X_test)[:, 1]
    print(f"fold {f}: train rows={train_mask.sum():,}  test rows={test_mask.sum():,}")

# baseline score, w04's exact rule, recomputed here
visible = model_df["impressions_90d"] >= 500
position_ok = (model_df["avg_position_90d"] > 0) & (model_df["avg_position_90d"] <= 20)
ctr_low = model_df["ctr"] < 0.4
declining = model_df["is_declining"] == 1
flag = visible & position_ok & ctr_low & declining
baseline_score = np.where(flag, model_df["impressions_90d"], 0)
print(f"\nBaseline flagged: {flag.sum():,} / {len(model_df):,}")

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

rows = []
for k in [20, 50, 100, 200]:
    row = {"k": k, "base_rate": round(y.mean(), 3), "baseline": round(precision_at_k(y, baseline_score, k), 3)}
    for name in model_builders:
        row[name] = round(precision_at_k(y, oof_scores[name], k), 3)
    rows.append(row)

comparison_table = pd.DataFrame(rows).set_index("k")
comparison_table



fold 0: train rows=208,280  test rows=52,005
fold 1: train rows=208,220  test rows=52,065
fold 2: train rows=208,120  test rows=52,165
fold 3: train rows=208,287  test rows=51,998
fold 4: train rows=208,233  test rows=52,052

Baseline flagged: 22,971 / 260,285


,base_rate,baseline,logistic_regression,decision_tree,random_forest
k,,,,,
20,0.289,0.60,0.30,0.800,0.900
50,0.289,0.62,0.42,0.720,0.920
100,0.289,0.61,0.43,0.660,0.860
200,0.289,0.62,0.41,0.685,0.845


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd

playbook_df = model_df.copy()
playbook_df["priority_score"] = oof_scores["decision_tree"]

def reason_code(row):
    if row["impressions_90d"] < 12:
        return "very_low_visibility_shortcut"  # the w05/w06 caveat: near-zero counts trip the label mechanically
    if row["ctr"] < 0.4 and 0 < row["avg_position_90d"] <= 20:
        return "declining_ctr_gap"  # matches w04's confirmed CTR-vs-position signal
    return "declining_low_confidence"

playbook_df["reason_code"] = playbook_df.apply(reason_code, axis=1)
playbook_df["action"] = np.where(playbook_df["priority_score"] >= 0.5, "refresh_review", "monitor")

queue_cols = ["content_hash_id", "client_hash_id", "priority_score", "reason_code", "action",
              "impressions_90d", "avg_position_90d", "ctr", "imp_early", "imp_late", "content_type"]
ranked_queue = playbook_df.sort_values("priority_score", ascending=False).reset_index(drop=True)[queue_cols]

print(f"Rows in queue: {len(ranked_queue):,}")
print(f"Flagged for refresh_review: {(ranked_queue['action'] == 'refresh_review').sum():,}")
print(ranked_queue["reason_code"].value_counts())
ranked_queue.head(20)

Rows in queue: 260,285
Flagged for refresh_review: 125,958
reason_code
very_low_visibility_shortcut    143433
declining_ctr_gap                64821
declining_low_confidence         52031
Name: count, dtype: int64


,content_hash_id,client_hash_id,priority_score,reason_code,action,impressions_90d,avg_position_90d,ctr,imp_early,imp_late,content_type
0,content_c675aad61bfd366c,client_3197e6291363b4db,0.933462,very_low_visibility_shortcut,refresh_review,8.0,49.833333,0.000000,0.0,8.0,keyword article
1,content_3b990ad54db3e543,client_3197e6291363b4db,0.933462,very_low_visibility_shortcut,refresh_review,7.0,4.500000,0.000000,4.0,3.0,keyword article
2,content_d7fdb2e2e374bf49,client_3197e6291363b4db,0.933462,very_low_visibility_shortcut,refresh_review,6.0,9.125000,0.000000,0.0,6.0,keyword article
3,content_e08c8b7a7186a5e5,client_3197e6291363b4db,0.933462,very_low_visibility_shortcut,refresh_review,1.0,19.000000,0.000000,0.0,1.0,keyword article
4,content_29b94cf89363222e,client_3197e6291363b4db,0.933462,declining_ctr_gap,refresh_review,23.0,9.126667,0.000000,5.0,18.0,keyword article
5,content_5f3870bf8125f0e7,client_3197e6291363b4db,0.933462,very_low_visibility_shortcut,refresh_review,1.0,12.000000,0.000000,0.0,1.0,keyword article
6,content_1985298f4023a8c9,client_3197e6291363b4db,0.933462,declining_low_confidence,refresh_review,62.0,67.480729,0.000000,1.0,61.0,keyword article
7,content_65ab0d22fcf556e3,client_3197e6291363b4db,0.933462,very_low_visibility_shortcut,refresh_review,2.0,2.500000,0.000000,0.0,2.0,keyword article
8,content_5f68cd109b5a2b4e,client_3197e6291363b4db,0.933462,declining_low_confidence,refresh_review,248.0,25.802482,0.000000,86.0,162.0,keyword article
9,content_acf9e20754cbb522,client_3197e6291363b4db,0.933462,declining_ctr_gap,refresh_review,20168.0,7.802624,0.044625,9828.0,10340.0,keyword article


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

- **Who:** a content reviewer deciding which pages to look at first in a weekly refresh cycle — a prioritization aid, not an automated action.
- **What it's for:** surfacing candidates worth a human look, ranked by a mix of observed decline and CTR-vs-position gap.
- **Where it stops being valid:**
  - Content types outside `keyword article` — the `w04` baseline's top 20 were 100% one content type; this queue hasn't been checked against others.
  - Any single client dominating the top of the queue — `w04` and `w05` both found 2-4 clients taking the majority of top picks (different clients each time), so a real handoff needs a per-client cap, not raw rank order.
  - Rows flagged `very_low_visibility_shortcut` — per `w05`/`w06`'s finding, near-zero-impression rows can trip the model's decision mechanically rather than through a real content signal. These need a human sanity check before any refresh work is scheduled on them.
  - Anything past `2026-04-30` — the model was trained on data through that date; it says nothing about what's true today.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Before acting on a `refresh_review` flag, a person should check:**
- Is there a SERP feature (featured snippet, People Also Ask) likely absorbing clicks? `avg_position_90d` measures a slot, not real visibility to a human searcher — this was the exact caveat on two of `w04`'s top-20 outliers.
- Does the page's `imp_late` collapse look like a technical/indexing issue rather than a content problem? (`w04`'s row 6 — impressions collapsed 279,427 → 250 within the window — was flagged as needing a technical check first, not a rewrite.)
- Is this one of the small number of clients dominating the queue? If so, confirm the client's own priorities before assuming this is genuinely their most urgent page.

**Should never be automated:**
- Auto-publishing or auto-editing content based on the score alone — this is a review queue, not a content generator.
- Treating `priority_score` as a confidence percentage — it's a rank, not a calibrated probability; `w05`/`w06` showed the score leans on a mechanical shortcut, so a high score is a "look here first," not "this is definitely declining."

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

- **Data drift:** if the mix of `content_type` in new content shifts away from `keyword article` (the type this whole pipeline was built and checked on), the rule and model should be re-validated before trusting them on the new mix.
- **Client mix drift:** if the set of active clients changes meaningfully (new large accounts, `gsc_data_start` shifting), the size-balanced fold assignment from `w05` needs to be rebuilt — it was computed for this specific 41-client population.
- **Time decay:** the feature window ends `2026-04-30`. A reasonable trigger: re-run the whole pipeline once real data extends ~90 days past that (i.e., once `2026-07-31`-era data is available), rather than continuing to score new content against an aging window.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
os.makedirs("../outputs", exist_ok=True)

ranked_queue.to_csv("../outputs/w07_action_playbook_queue.csv", index=False)
print(f"Wrote {len(ranked_queue):,} rows to work/outputs/w07_action_playbook_queue.csv")

# Summary table for the paper: baseline vs model, same shape as w05's comparison_table
summary = comparison_table.copy() if "comparison_table" in dir() else None
if summary is not None:
    summary.to_csv("../outputs/w05_comparison_table.csv")
    print("Also wrote work/outputs/w05_comparison_table.csv")

Wrote 260,285 rows to work/outputs/w07_action_playbook_queue.csv
Also wrote work/outputs/w05_comparison_table.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.